# 623 direct SPP action prediction: stateful LSTM vs three-step CNN

This notebook trains direct source-interface students, not an SPP candidate gate. Both models consume only the causal line-aligned address sequence and independently predict 64 same-page offsets × `FILL_L2/FILL_LLC`. Captured SPP actions are train labels and an evaluation fidelity audit, never model inputs. The CNN has exactly one causal `Conv1d` with a three-callback moving window.

In [ ]:
import hashlib, os, pathlib, shutil, subprocess, sys, tarfile, torch
from google.colab import userdata
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU (A100 preferred)'
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(torch.cuda.get_device_name(0), torch.__version__)

REPO = '/content/cache_arch'
PUBLIC_URL = 'https://github.com/Angelawoo572/cache_arch.git'
TOKEN = userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add GITHUB_TOKEN in Colab Secrets'
ASKPASS = '/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo x-access-token ;;\n  *) echo "$GITHUB_TOKEN" ;;\nesac\n')
os.chmod(ASKPASS, 0o700)
git_env = os.environ.copy()
git_env.update({'GIT_ASKPASS': ASKPASS, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': TOKEN})
try:
    if not os.path.isdir(REPO):
        subprocess.run(['git', 'clone', PUBLIC_URL, REPO], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only', 'origin', 'main'], check=True, env=git_env)
finally:
    pathlib.Path(ASKPASS).unlink(missing_ok=True)
print('Git:', subprocess.check_output(['git', '-C', REPO, 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_ID = '623_offline_lstm_cnn_spp_direct_seed7'
DRIVE_ROOT = f'/content/drive/MyDrive/cache_prefetch_623_spp_direct/{RUN_ID}'
INPUT_DIR = f'{DRIVE_ROOT}/colab_input'
OUTPUT_ROOT = f'{DRIVE_ROOT}/colab_output'
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_ROOT, exist_ok=True)
INPUT_ARCHIVE = f'{DRIVE_ROOT}/{RUN_ID}.colab_input.tar.gz'
REUPLOAD_INPUT = True
if REUPLOAD_INPUT or not os.path.isfile(INPUT_ARCHIVE):
    from google.colab import files
    expected = f'{RUN_ID}.colab_input.tar.gz'
    uploaded = files.upload()
    assert expected in uploaded, f'Select {expected}; got {list(uploaded)}'
    pathlib.Path(INPUT_ARCHIVE).write_bytes(uploaded[expected])
if os.path.isdir(INPUT_DIR): shutil.rmtree(INPUT_DIR)
os.makedirs(INPUT_DIR, exist_ok=True)
with tarfile.open(INPUT_ARCHIVE, 'r:gz') as archive:
    archive.extractall(INPUT_DIR)
for record in pathlib.Path(f'{INPUT_DIR}/SHA256SUMS').read_text().splitlines():
    expected_sha, filename = record.split(maxsplit=1)
    filename = filename.lstrip('*')
    observed_sha = hashlib.sha256(pathlib.Path(f'{INPUT_DIR}/{filename}').read_bytes()).hexdigest()
    assert observed_sha == expected_sha, f'archive SHA256 mismatch: {filename}'
print('Input:', INPUT_ARCHIVE)

In [ ]:
import gzip, json
TRACE = '623.xalancbmk_s-700B'
POLICY = 'spp'
ROLES = ('train', 'guard', 'eval')
INPUTS = {}
for role in ROLES:
    stream = f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_stream.csv.gz'
    actions = f'{INPUT_DIR}/{TRACE}.{POLICY}.{role}_teacher_actions.csv.gz'
    INPUTS[role] = {'stream': stream, 'teacher_actions': actions}
    for path in (stream, actions):
        assert os.path.isfile(path), path
        digest = hashlib.sha256()
        with gzip.open(path, 'rb') as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b''):
                digest.update(block)
        print(path, digest.hexdigest())
COLLECTION_MANIFEST = f'{INPUT_DIR}/collection_manifest.json'
SOURCE_CONTRACT_INPUT = f'{INPUT_DIR}/spp_source_contract.json'
collection = json.loads(pathlib.Path(COLLECTION_MANIFEST).read_text())
assert collection['status'] == 'PASS'
assert collection['experiment_revision'] == 'spp_direct_io_sliding_cnn_v2'
assert collection['event_logger_schema'] == '623_causal_trigger_v5'
assert collection['action_attachment_mode'] == 'explicit_trigger_event_id'
assert collection['neural_role'] == 'direct_spp_action_predictor'
assert collection['model_input_is_causal_address_sequence_only'] is True
assert collection['teacher_actions_are_model_inputs'] is False
assert collection['nn_can_generate_actions_not_emitted_by_teacher'] is True
SCRIPT = f'{REPO}/formal_NN_training/experiments/623_offline_lstm_cnn_spp/python/train_and_offline_infer.py'
CONTRACT = f'{REPO}/formal_NN_training/experiments/623_offline_lstm_cnn_spp/data/stream_contract.json'
SOURCE_CONTRACT_REPO = f'{REPO}/formal_NN_training/experiments/623_offline_lstm_cnn_spp/data/spp_source_contract.json'
for path in (SCRIPT, CONTRACT, SOURCE_CONTRACT_INPUT, SOURCE_CONTRACT_REPO): assert os.path.isfile(path), path
assert hashlib.sha256(pathlib.Path(SOURCE_CONTRACT_INPUT).read_bytes()).hexdigest() == hashlib.sha256(pathlib.Path(SOURCE_CONTRACT_REPO).read_bytes()).hexdigest()
contract = json.loads(pathlib.Path(CONTRACT).read_text())
assert contract['experiment_revision'] == 'spp_direct_io_sliding_cnn_v2'
assert contract['direct_action_space']['classes'] == 128
assert contract['direct_action_space']['teacher_actions_are_model_inputs'] is False
assert contract['cnn']['temporal_convolution_layers'] == 1
assert contract['cnn']['kernel_size_events'] == 3
assert contract['cnn']['stride_events'] == 1
assert contract['cnn']['additional_temporal_layers'] == 0
print(json.dumps(contract['architecture_pairs'], indent=2))

In [ ]:
# Train on local Colab disk; copy only durable outputs to Drive.
LOCAL_INPUT = f'/content/{RUN_ID}_colab_input'
LOCAL_OUTPUT = f'/content/{RUN_ID}_colab_output'
for path in (LOCAL_INPUT, LOCAL_OUTPUT):
    if os.path.isdir(path): shutil.rmtree(path)
shutil.copytree(INPUT_DIR, LOCAL_INPUT)
MODEL_SPECS = [
    {'tag': 'direct_spp_lstm_h4',  'family': 'lstm', 'size': 4,  'pair_id': 'p0', 'parameters': 880},
    {'tag': 'direct_spp_cnn_c5',   'family': 'cnn',  'size': 5,  'pair_id': 'p0', 'parameters': 908},
    {'tag': 'direct_spp_lstm_h8',  'family': 'lstm', 'size': 8,  'pair_id': 'p1', 'parameters': 1760},
    {'tag': 'direct_spp_cnn_c10',  'family': 'cnn',  'size': 10, 'pair_id': 'p1', 'parameters': 1688},
    {'tag': 'direct_spp_lstm_h16', 'family': 'lstm', 'size': 16, 'pair_id': 'p2', 'parameters': 3904},
    {'tag': 'direct_spp_cnn_c24',  'family': 'cnn',  'size': 24, 'pair_id': 'p2', 'parameters': 3872},
]
SWEEP = []
local_inputs = {role: {kind: f'{LOCAL_INPUT}/{TRACE}.{POLICY}.{role}_{kind}.csv.gz' for kind in ('stream', 'teacher_actions')} for role in ROLES}
local_source_contract = f'{LOCAL_INPUT}/spp_source_contract.json'
for spec in MODEL_SPECS:
    tag = spec['tag']
    local_out = f'{LOCAL_OUTPUT}/{tag}'
    drive_out = f'{OUTPUT_ROOT}/{tag}'
    if os.path.isdir(local_out): shutil.rmtree(local_out)
    if os.path.isdir(drive_out): shutil.rmtree(drive_out)
    cmd = [sys.executable, SCRIPT, '--policy', POLICY]
    for role in ROLES:
        cmd += [f'--{role}-stream', local_inputs[role]['stream']]
        cmd += [f'--{role}-teacher-actions', local_inputs[role]['teacher_actions']]
    cmd += ['--source-contract', local_source_contract, '--out-dir', local_out, '--model-family', spec['family'], '--model-size', str(spec['size']), '--pair-id', spec['pair_id'], '--device', 'cuda', '--seed', '7', '--epochs', '8', '--chunk-len', '1024', '--accumulate-chunks', '16']
    print('\nTraining', tag, 'command:', ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout: print(result.stdout, end='')
    if result.returncode != 0:
        if result.stderr: print(result.stderr, end='')
        raise RuntimeError(f'{tag} failed with exit code {result.returncode}')
    metadata = json.loads(pathlib.Path(f'{local_out}/run_metadata.json').read_text())
    assert metadata['model_tag'] == tag
    assert metadata['parameter_count'] == spec['parameters']
    assert metadata['matched_normal_prefetcher'] == POLICY
    assert metadata['neural_role'] == 'direct_spp_action_predictor'
    assert metadata['model_input_is_causal_address_sequence_only'] is True
    assert metadata['teacher_actions_are_model_inputs'] is False
    assert metadata['nn_can_generate_actions_not_emitted_by_teacher'] is True
    assert metadata['model_does_not_use_pc'] is True
    assert metadata['training_chunks_shuffled'] is False
    assert metadata['event_logger_schema'] == '623_causal_trigger_v5'
    assert metadata['action_attachment_mode'] == 'explicit_trigger_event_id'
    assert metadata['experiment_revision'] == 'spp_direct_io_sliding_cnn_v2'
    assert metadata['causal_no_future_self_test'] == 'PASS'
    assert metadata['cnn_architecture_self_test'] == 'PASS'
    if spec['family'] == 'cnn':
        assert metadata['cnn_temporal_layers'] == 1
        assert metadata['cnn_kernel_size'] == 3
        assert metadata['cnn_stride'] == 1
        assert metadata['cnn_dilation'] == 1
    shutil.copytree(local_out, drive_out)
    SWEEP.append({key: metadata[key] for key in ('model_tag', 'model_family', 'model_size', 'architecture_pair_id', 'parameter_count', 'threshold', 'offline_normal_entries', 'offline_nn_entries', 'offline_normal_fill_level_counts', 'offline_nn_fill_level_counts', 'eval_action_fidelity')})
manifest = {'trace': TRACE, 'revision': 'spp_direct_io_sliding_cnn_v2', 'event_logger_schema': '623_causal_trigger_v5', 'action_attachment_mode': 'explicit_trigger_event_id', 'points': SWEEP}
pathlib.Path(f'{LOCAL_OUTPUT}/sweep_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
shutil.copy2(f'{LOCAL_OUTPUT}/sweep_manifest.json', f'{OUTPUT_ROOT}/sweep_manifest.json')
print(json.dumps(SWEEP, indent=2))

In [ ]:
for point in SWEEP:
    out_dir = f"{OUTPUT_ROOT}/{point['model_tag']}"
    required = ['offline_spp.replay.csv', 'offline_nn.replay.csv', 'model.pt', 'run_metadata.json', 'policy_sweep.csv']
    assert all(os.path.isfile(f'{out_dir}/{name}') for name in required), out_dir
OUTPUT_ARCHIVE = f'{DRIVE_ROOT}/{RUN_ID}.colab_output.tar.gz'
LOCAL_ARCHIVE = f'/content/{RUN_ID}.colab_output.tar.gz'
with tarfile.open(LOCAL_ARCHIVE, 'w:gz') as archive:
    for item in pathlib.Path(LOCAL_OUTPUT).iterdir():
        archive.add(item, arcname=item.name)
shutil.copy2(LOCAL_ARCHIVE, OUTPUT_ARCHIVE)
print('DONE:', OUTPUT_ARCHIVE, os.path.getsize(OUTPUT_ARCHIVE), 'bytes')

After copying the output archive to the server, run this independent SPP directory's `replay` stage. The analyzer compares every direct NN list with fill-preserving `offline_spp` and reports IPC, L2 miss rate, selected accuracy, coverage, timeliness, BPI, and direct action fidelity.